## Upload data to local Postgre SQL database

Cálculo de la ampacidad IEEE y CIGRE con la implementación UC. El script está diseñado para trabajar con los ficheros .csv descargados de la web en la nube de GE [pestaña dedicada a REPORTS] agrupados por línea y definidos con fechas del 1 [00:00] al último día del mes [23:59], que deben tener un nombre con la estructura siguiente:

Apoyo XXXXX AAMM.csv   
XXXXX .- nombre del apoyo  
AA .- Año  
MM .- Mes  


Las columnas del fichero serán:

Date;Time;Temperature;Windspeed AVG;Wind Direction AVG;Wind Direction Dominant;Solar Radiation;Dew Point Temperature;WindSonic Windspeed AVG;WindSonic Wind Direction AVG;WindSonic Wind Direction Dominant;I;T;RCC IEEE;RCC CIGRE;CLEARANCE;RIME


Ordenarlos en carpetas atendiendo al nombre de la línea y definir los arrays y variable siguientes antes de ejecutar:   

patha = ['e:/GE3/EL PALMAR-ESPINARDO 1/', 'e:/GE3/HELLIN-CALASPARRA/']   
LineNamea = ['El Palmar-Espinardo', 'Hellín-Calasparra']   
table_name    


**author**: GTEA-UC  
**email**: mananam@unican.es  
**last update**: 08/12/2023

In [51]:
import sys 
print( sys.version)

3.9.15 (main, Nov 24 2022, 14:39:17) [MSC v.1916 64 bit (AMD64)]


In [52]:
import psycopg2
import pyodbc
import configparser # Config file
import pandas as pd
import datetime
from datetime import datetime, timedelta
import calendar
import time
from dateutil.relativedelta import relativedelta
from sqlalchemy import create_engine
import os

**Paquetes específicos DLR**

In [53]:
from cable import cable
from case import case
from ieee738 import ieee738
from cigre601 import cigre601
from pvsystems import pvsystems
import matplotlib.pyplot as plt 


# Needed only during the development phase.
from importlib import reload
reload( cable)
reload( case)
reload( ieee738)
reload( cigre601)
reload( pvsystems)

<module 'pvsystems.pvsystems' from 'e:\\mario\\python\\pypacity\\pvsystems\\pvsystems.py'>

Upload the dataframe to the database table

Information for the connection to the database

In [54]:
# PostgreSQL connection parameters
db_params = {
    'host': 'localhost', # the database is in the same machine where this script is running
    'database': 'Iberdrola', # name of the database
    'user': 'postgres', # user
    'password': 'mananam05' # password
}

Create a connection to the database. 

Define the name of the table where the transactions will take place.

In [55]:
# Establish a connection to the PostgreSQL database using SQLAlchemy
connection_string = f"postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params['host']}/{db_params['database']}"
engine = create_engine(connection_string)

# Define the name of the table you want to create or replace
table_name = 'ucgeproc2'

patha .- is the path to the folder where the information about the lines is storaged.

LineName .- is the array with the names of the lines. This array is defined according patha


In [56]:
patha = ['e:/GE3/EL PALMAR-ESPINARDO 1/', 'e:/GE3/HELLIN-CALASPARRA/']
LineNamea = ['EL PALMAR - ESPINARDO', 'HELLÍN - CALASPARRA']

#patha = ['e:/GE3/HC/', 'e:/GE3/PE/']
#LineNamea = ['Hellín-Calasparra', 'El Palmar-Espinardo']

#patha = [ 'e:/GE3/PE/']
#LineNamea = [ 'El Palmar-Espinardo']

file_case = "E:/mario/trabajos2/iberdrola_DTR/datos/case.xlsx" # Information about the cases
file_cable = "E:/mario/trabajos2/iberdrola_DTR/datos/cable.xlsx" # Information about the cables

dfCases = pd.read_excel( file_case, header=0)
dfCables = pd.read_excel( file_cable, header=0)

In [57]:
dfCases

,LineName,NodeName,ANG_DEG,CDR_ELEV,Z1_DEG,CDR_LAT_DEG,TCDR,ALBEDO
0,El Palmar-Espinardo,10160,0,50.17,0,38,50,0.2
1,El Palmar-Espinardo,10166,0,46.00,0,38,50,0.2
2,El Palmar-Espinardo,10174,0,45.86,0,38,50,0.2
3,Hellín-Calasparra,10021,0,336.00,0,38,60,0.2
4,Hellín-Calasparra,10031,0,459.00,0,38,60,0.2
5,Hellín-Calasparra,10042,0,358.00,0,38,60,0.2
6,Hellín-Calasparra,10055,0,438.00,0,38,60,0.2
7,Hellín-Calasparra,10068,0,479.00,0,38,60,0.2
8,Hellín-Calasparra,10085,0,579.00,0,38,60,0.2


In [58]:
dfCables

,LineName,Cstring,D,d,TLO,THI,TCDRMAX,RLO,RHI,EMISS,...,HNH,HEATOUT,HEATCORE,TotalS,CSteel20,CAlum20,BetaSteel20,BetaAlum20,mSteel,mAlum
0,El Palmar-Espinardo,LA-280,0.0218,0.0034,20,75,50,0.000119,0.000146,0.5,...,2,NaN,NaN,241.7,481,897,0.0001,0.000318,0.3977,0.2641
1,Hellín-Calasparra,LA-180,0.0175,0.0025,20,75,65,0.000196,0.000241,0.8,...,2,NaN,NaN,147.3,481,897,0.0001,0.000318,0.3977,0.2641


In [59]:
def get_df( data_frame, LineName, NodeName, dfCable, dfCase):
    
    column_names = [
            'LineName',
            'NodeName',
            'TimeStamp',
            'PhaseCurrent',
            'PhasePhase' ,
            'ConductorTemp' ,
            'AmbientTemp' ,
            'WindSpeed' ,
            'WindDomDirection' ,
            'WindAvgDirection' ,
            'SolarRadiation' ,
            'DewPoint' ,
            'ServiceName' ,
            'Clearance' ,
            'IMAX' , 
            'LoadMVA' ,
            'MaxCapacityMVA' ,
            'IEEE738' ,
            'CIGRE601' ,
            'ucIEEE738' ,
            'ucCIGRE601' ]
    
    df2 = pd.DataFrame( columns = column_names) # define a new dataframe with the columns defined by the array column_names.
    for index, row in data_frame.iterrows():
        
        LineName = str(LineName)
        NodeName = NodeName
        datetime_str = str(row['Date']) + "  " + str(row['Time'])
        datetime_obj = datetime.strptime( datetime_str, "%d/%m/%Y %H:%M")
        TimeStamp = datetime_obj.strftime('%Y-%m-%d %H:%M:%S') 
        print('\r' + TimeStamp, end=' ', flush=True)
        
        Temperature = float(row['Temperature'])
        WindspeedAVG = float(row['Windspeed AVG'])
        WindDirectionAVG = float(row['Wind Direction AVG'])
        WindDirectionDominat = float(row['Wind Direction Dominant'])
        SolarRadiation = float(row['Solar Radiation'])
        DewPointTemperature = float(row['Dew Point Temperature'])
        WindSonic = float(row['WindSonic Windspeed AVG'])
        WindSonicWindDirectionAVG = float(row['WindSonic Wind Direction AVG'])
        WindSonicWindDirectionDominant = float(row['WindSonic Wind Direction Dominant'])
        PhaseCurrent = float(row['I'])
        ConductorTemp = float(row['T'])
        RCCIEEE = float(row['RCC IEEE'])
        RCCCIGRE = float(row['RCC CIGRE'])
        Clearance = float(row['CLEARANCE'])
        RIME = float(row['RIME'])
    

        NSELECT = 2 
        Cable1 = cable.Cable()
        #c_db, error = Cable1.load_cable_db()
        #Cable1.set_cable( NSELECT, conductor = dfCable.at[1,'Cstring'])

        Cable1.D = 1000*float( dfCable['D'])
        #Cable1.C = 10.4
        Cable1.d = 1000*float( dfCable['d']) 
        Cable1.TLO = float( dfCable['TLO']) 
        Cable1.THI = float( dfCable['THI']) 
        #Cable1.TCDRMAX = 60.0
        Cable1.RLO = float( dfCable['RLO']) 
        Cable1.RHI = float( dfCable['RHI']) 

        Cable1.EMISS = float( dfCable['EMISS'])
        Cable1.ABSORP = float( dfCable['ABSORP'])


        Cable1.HNH = int( dfCable['HNH']) 
        #Cable1.HEATOUT = 357.9
        #Cable1.HEATCORE = 132.1
        Cable1.TotalS = float( dfCable[ 'TotalS']) 
        Cable1.CSteel20 = float( dfCable['CSteel20']) 
        Cable1.CAlum20 = float( dfCable[ 'CAlum20']) 
        Cable1.BetaSteel20 = float( dfCable[ 'BetaSteel20']) 
        Cable1.BetaAlum20 = float( dfCable[ 'BetaAlum20']) 
        Cable1.mSteel = float( dfCable[ 'mSteel']) 
        Cable1.mAlum = float( dfCable[ 'mAlum']) 


        Case1 = case.Case()
        Case1.demo( NSELECT)
        # Ambient conditions
        Case1.TAMB = Temperature
        Case1.CDR_LAT_DEG = float(dfCase['CDR_LAT_DEG'])
        Case1.ALBEDO = float(dfCase['ALBEDO'])
        Case1.beta = 0
        Case1.CDR_ELEV = float(dfCase['CDR_ELEV'])
        Case1.TCDRPRELOAD = float( dfCase['TCDR'])
        #Case1.TCDRMAX = 150
        #Case1.TCDR = 100.0
        Case1.SolarRadiation = SolarRadiation
        Case1.VWIND = WindspeedAVG + 0.01
        Case1.WINDANG_DEG = abs(WindDirectionAVG-float( dfCase['ANG_DEG']))
        Case1.Z1_DEG = float(dfCase['Z1_DEG'])
        Case1.Ns = 1.0
        Case1.SUN_TIME = 99 # solar Radiation measured (IEEE738)
        Case1.SOLAR = 0 # Solar Radiation measured (CIGRE601)
        dia = int(datetime_obj.strftime('%d'))
        mes = int(datetime_obj.strftime('%m') )
        Case1.NDAY = PV1.DayOfYear( dia, mes) # 10th June
        #print("NDAY: " + str(Case1.NDAY))

        # IEEE 738
        X1 = ieee738.IEEE738()
        X1.Debug = 0
        X1.set_cable( Cable1)
        X1.set_case( Case1)
        X1.Case1.SORM = 1
        X1.ieee_738_2013()      
        #X1.output()
        ucIEEE738 = round( float(X1.Case1.TR), 2)

        X2 = cigre601.CIGRE601()
        X2.Debug = 0
        X2.set_cable( Cable1)
        X2.set_case( Case1)
        X2.cigre601()    
        #X2.output()
        ucCIGRE601 =round( float(X2.Case1.TR), 2)
    
    
        new_row_data = {
            'LineName' : LineName,
            'NodeName' : NodeName,
            'TimeStamp' : TimeStamp,
            'PhaseCurrent' : PhaseCurrent,
            'PhasePhase' : ' ',
            'ConductorTemp' : ConductorTemp,
            'AmbientTemp' : Temperature,
            'WindSpeed' : WindspeedAVG,
            'WindDomDirection' : WindDirectionDominat,
            'WindAvgDirection' : WindDirectionAVG,
            'SolarRadiation' : SolarRadiation,
            'DewPoint' : DewPointTemperature,
            'ServiceName' : ' ',
            'Clearance' : Clearance,
            'IMAX' : ' ',
            'LoadMVA' : ' ',
            'MaxCapacityMVA' : ' ',
            'IEEE738' : RCCIEEE,
            'CIGRE601' : RCCCIGRE,
            'ucIEEE738' : ucIEEE738,
            'ucCIGRE601' : ucCIGRE601}

        new_row_type = {
            'LineName' : str,
            'NodeName' : str,
            'TimeStamp' : str,
            'PhaseCurrent' : float, 
            'PhasePhase' : float, 
            'ConductorTemp' : float, 
            'AmbientTemp' : float, 
            'WindSpeed' : float, 
            'WindDomDirection' : float, 
            'WindAvgDirection' : float, 
            'SolarRadiation' : float, 
            'DewPoint' : float, 
            'ServiceName' : str,
            'Clearance' : Clearance,
            'IMAX' : float,
            'LoadMVA' : float, 
            'MaxCapacityMVA' :  float, 
            'IEEE738' : float, 
            'CIGRE601' : float, 
            'ucIEEE738' : float, 
            'ucCIGRE601' : float}




        new_df = pd.DataFrame( [new_row_data]) #, dtype=new_row_data)
        df2 = pd.concat( [df2, new_df], ignore_index=True)
        
    return( df2)   
    

In [60]:
dtype_mapping = {
            'LineName' : 'text',
            'NodeName' : 'text',
            'TimeStamp' : 'timestamp',
            'PhaseCurrent' : 'double precision',
            'PhasePhase' : 'double precision',
            'ConductorTemp' : 'double precision',
            'AmbientTemp' : 'double precision',
            'WindSpeed' : 'double precision',
            'WindDomDirection' : 'double precision',
            'WindAvgDirection' : 'double precision',
            'SolarRadiation' : 'double precision',
            'DewPoint' : 'double precision',
            'ServiceName' : 'text',
            'Clearance' : 'double precision',
            'IMAX' : 'double precision',
            'LoadMVA' : 'double precision',
            'MaxCapacityMVA' : 'double precision',
            'IEEE738' : 'double precision',
            'CIGRE601' : 'double precision',
            'ucIEEE738' : 'double precision',
            'ucCIGRE601' : 'double precision'}


for index_p, value_p in enumerate( patha):
    file_list = os.listdir( patha[index_p])
    print( file_list)
    PV1 = pvsystems.PVSystems()
    
    for index, value in enumerate( file_list):
        LineName = LineNamea[ index_p]
        split_list = value.split(" ")
        NodeName = split_list[1]
        
        dfCable = dfCables[ dfCables['LineName']==LineName]
        print("***************************************")
        print(dfCable)
        dfCase = dfCases[ dfCases['NodeName']==int(NodeName)]
        print("-----------------------------------------")
        print(dfCase)
        print("***************************************")

       
        
        #print( NodeName)
        YearMonth_list = split_list[2].split(".") 
        YearMonth = YearMonth_list[0]
        #print( YearMonth)
        y1 = YearMonth[0]
        y2 = YearMonth[1]
        yy = y1 + y2
        year = 2000 + int( yy)
        #print(year)
        m1 = YearMonth[2]
        m2 = YearMonth[3]
        mm = m1 + m2
        month = int(mm)
        fullpath = patha[index_p] + file_list[index]
        print( "%s ### %s - %s. %i / %i" %(fullpath, LineName, NodeName,  year, month))
        data_frame2 = pd.read_csv( fullpath, delimiter=';', header=0, decimal=',')
        
        # Filter colums with missing values
        columns_to_check = [
            'Date', 
            'Time', 
            'Temperature', 
            'Windspeed AVG', 
            'Wind Direction AVG',
            'Wind Direction Dominant', 
            'Solar Radiation', 
            'Dew Point Temperature',
            'I', 
            'T' 
            ]
        data_frame3 = data_frame2.dropna( subset=columns_to_check)
        
        df22 = get_df( data_frame3, LineName, NodeName, dfCable, dfCase)
        print(df22)
        #df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid', dtype=dtype_mapping) 
        df22.to_sql( table_name, engine, if_exists='append', index=True, index_label='geprocid') 
        

['Apoyo 10160 2301.csv']
***************************************
              LineName Cstring       D       d  TLO  THI  TCDRMAX       RLO  \
0  El Palmar-Espinardo  LA-280  0.0218  0.0034   20   75       50  0.000119   

        RHI  EMISS  ...  HNH  HEATOUT  HEATCORE  TotalS  CSteel20  CAlum20  \
0  0.000146    0.5  ...    2      NaN       NaN   241.7       481      897   

   BetaSteel20  BetaAlum20  mSteel   mAlum  
0       0.0001    0.000318  0.3977  0.2641  

[1 rows x 21 columns]
-----------------------------------------
              LineName  NodeName  ANG_DEG  CDR_ELEV  Z1_DEG  CDR_LAT_DEG  \
0  El Palmar-Espinardo     10160        0     50.17       0           38   

   TCDR  ALBEDO  
0    50     0.2  
***************************************
e:/GE3/PE/Apoyo 10160 2301.csv ### El Palmar-Espinardo - 10160. 2023 / 1
2023-01-02 03:44:00                                                                                                      LineName NodeName            TimeStamp  P

In [61]:
print(df22.dtypes)

LineName             object
NodeName             object
TimeStamp            object
PhaseCurrent        float64
PhasePhase           object
ConductorTemp       float64
AmbientTemp         float64
WindSpeed           float64
WindDomDirection    float64
WindAvgDirection    float64
SolarRadiation      float64
DewPoint            float64
ServiceName          object
Clearance           float64
IMAX                 object
LoadMVA              object
MaxCapacityMVA       object
IEEE738             float64
CIGRE601            float64
ucIEEE738           float64
ucCIGRE601          float64
dtype: object


In [62]:
# Close the database connection
engine.dispose()